# 5.1 Ứng dụng BPTT trong Language Modeling

## 1. Bài toán Language Modeling là gì?

**Language modeling** (mô hình hóa ngôn ngữ) là bài toán dự đoán từ kế tiếp trong một chuỗi từ. Cụ thể, mô hình sẽ học phân phối xác suất có điều kiện:

$$P(w_t|w_1, w_2, ..., w_{t-1})$$

Trong đó $( w_t )$ là từ tại thời điểm $( t )$, và mô hình có gắng học cách sinh ra câu có ý nghĩa dựa trên ngữ cảnh từ trước.

## 2. Vì sao cần sử dụng RNN và BPTT?

Do ngôn ngữ mang tính tuần tự, RNN là kiến trúc phù hợp để xử lý chuỗi văn bản, vì nó có khả năng lưu giữ trạng thái qua các bước thời gian. Tuy nhiên, để huấn luyện RNN hiệu quả, ta cần **lan truyền lỗi ngược qua thời gian** – hay **Backpropagation Through Time (BPTT)** – nhằm cập nhật các trọng số tại mỗi bước thời gian dựa trên tổng lỗi tích lũy.

## 3. Vai trò của BPTT trong huấn luyện Language Model

- Trong quá trình huấn luyện, RNN được *unroll* (mở ra) qua nhiều bước thời gian, ví dụ 35 bước.
- Với mỗi batch văn bản, mô hình sinh ra các dự đoán tại mỗi bước $( t )$, và lỗi tổng hợp được tính trên toàn bộ chuỗi.
- Sau đó, BPTT sẽ **truyền lỗi ngược từ thời điểm** $( T )$ về 0, để tính gradient và cập nhật trọng số của mạng RNN.

BPTT là cốt lõi để RNN học được ngữ cảnh dài, giúp mô hình sinh ngôn ngữ tự nhiên hơn.

## 4. Cài đặt mô hình RNN Language Model sử dụng BPTT (PyTorch + GPU)

In [8]:
pip install datasets

   ---------------------------------------- 0.0/25.8 MB ? eta -:--:--
   --- ------------------------------------ 2.4/25.8 MB 12.2 MB/s eta 0:00:02
   ------- -------------------------------- 5.0/25.8 MB 12.1 MB/s eta 0:00:02
   ----------- ---------------------------- 7.3/25.8 MB 11.9 MB/s eta 0:00:02
   --------------- ------------------------ 10.0/25.8 MB 11.9 MB/s eta 0:00:02
   ------------------- -------------------- 12.3/25.8 MB 11.9 MB/s eta 0:00:02
   ---------------------- ----------------- 14.7/25.8 MB 11.8 MB/s eta 0:00:01
   -------------------------- ------------- 17.0/25.8 MB 11.8 MB/s eta 0:00:01
   ------------------------------ --------- 19.7/25.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------- ----- 22.0/25.8 MB 11.8 MB/s eta 0:00:01
   -------------------------------------- - 24.6/25.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 25.8/25.8 MB 11.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.6 MB ? eta -:

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.5.1+cu121
True
12.1


## 3. Xử lý dữ liệu với HuggingFace datasets

In [9]:
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

# Tải tập dữ liệu WikiText2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
tokenizer = lambda x: x.split()

# Tạo vocabulary thủ công
from collections import Counter
counter = Counter()
for line in dataset["train"]["text"]:
    counter.update(tokenizer(line))

vocab = {word: i+2 for i, (word, _) in enumerate(counter.items())}
vocab["<unk>"] = 0
vocab["<pad>"] = 1
inv_vocab = {i: w for w, i in vocab.items()}

# Mã hóa dữ liệu
def encode(text):
    return torch.tensor([vocab.get(token, vocab["<unk>"]) for token in tokenizer(text)], dtype=torch.long)

# Nối toàn bộ văn bản thành 1 chuỗi dài
encoded_data = [encode(line) for line in dataset["train"]["text"] if line.strip()]
train_data = torch.cat(encoded_data)

# Tạo batch
def batchify(data, batch_size):
    n_batch = data.size(0) // batch_size
    data = data[:n_batch * batch_size]
    return data.view(batch_size, -1).t().contiguous()

batch_size = 20
bptt = 35
train_data = batchify(train_data, batch_size).to("cuda")

E:\Anaconda\envs\torchgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E:\Anaconda\envs\torchgpu\lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: ht

## 4. Tạo các minibatch và hàm lấy input-target

In [10]:
def get_batch(source, i):
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

## 5. Mô hình RNN đơn giản cho Language Modeling

In [11]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.num_layers = num_layers
        self.hidden_size = hidden_size

    def forward(self, x, hidden):
        emb = self.embed(x)
        out, hidden = self.rnn(emb, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size).to("cuda")

## 6. Huấn luyện mô hình sử dụng BPTT

In [12]:
vocab_size = len(vocab)
embed_size = 200
hidden_size = 200
num_layers = 2

model = RNNModel(vocab_size, embed_size, hidden_size, num_layers).to("cuda")
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=5.0)

# Huấn luyện
model.train()
hidden = model.init_hidden(batch_size)

for epoch in range(1):
    total_loss = 0.
    for batch, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
        data, targets = get_batch(train_data, i)
        optimizer.zero_grad()

        # BPTT: chạy forward toàn chuỗi
        output, hidden = model(data, hidden)

        # Ngắt gradient để tránh tính ngược quá sâu
        hidden = hidden.detach()

        loss = criterion(output.view(-1, vocab_size), targets)
        loss.backward()  # ← BPTT ở đây

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        total_loss += loss.item()
        if batch % 100 == 0 and batch > 0:
            print(f"Batch {batch}, Loss: {total_loss / 100:.2f}")
            total_loss = 0

AttributeError: module 'torch._functorch.eager_transforms' has no attribute 'grad_and_value'

In [13]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True

AttributeError: module 'torch._functorch.eager_transforms' has no attribute 'grad_and_value'

In [ ]:
def evaluate(model, data_source):
    model.eval()
    total_loss = 0.
    hidden = model.init_hidden(batch_size)
    with torch.no_grad():
        for i in range(0, data_source.size(0) - 1, bptt):
            data, targets = get_batch(data_source, i)
            output, hidden = model(data, hidden)
            hidden = hidden.detach()
            loss = criterion(output.view(-1, vocab_size), targets)
            total_loss += loss.item()
    return total_loss / ((data_source.size(0) - 1) // bptt)

val_loss = evaluate(model, train_data)
print(f"Validation loss: {val_loss:.2f}, Perplexity: {math.exp(val_loss):.2f}")